# IV - Triptyques, temps, lieux


Il produit un seul fichier :

```text
../results/csv_triptyques/triptyques_temps_lieux.csv
```

Règle du déplacement :

```text
déplacement = triptyque avec au moins un personnage et au moins un lieu
```


In [10]:
# Librairies utiles
import re
import ast
import bisect
import pandas as pd
import numpy as np

# Fichiers d'entrée et de sortie
MANUEL_TRIPLETS = "../results/csv_triptyques/manuel_chap1to5_chap.csv"
MANUEL_ENTITIES = "../data/SACR/all_annots.sacr.entities"
OUT_CSV = "../results/csv_triptyques/triptyques_temps_lieux.csv"

# Premier jour du voyage dans le roman
DATE_DEBUT_CALENDRIER = "1872-10-02"


## 1) Charger les fichiers

On part seulement des triptyques manuels et des entités manuelles, car les `COREF_name` temporels y sont déjà normalisés.


In [11]:
# Chargement des triptyques et des entités annotées manuellement
trip = pd.read_csv(MANUEL_TRIPLETS).copy()
ent_manuel = pd.read_csv(MANUEL_ENTITIES, sep="	")

# Identifiant simple pour retrouver chaque triptyque après export
trip["triplet_id"] = trip.index

print("Triptyques :", trip.shape)
print("Entités :", ent_manuel.shape)


Triptyques : (1255, 34)
Entités : (1229, 33)


## 2) Fonctions utiles

`parse_time_code` lit les codes calendaires du type `1872-10-02-11` ou `1872_10_02_11`.

Les codes incomplets comme `1872-00-00-00` sont gardés comme repères temporels, mais ils ne permettent pas de calculer le jour romanesque. Les codes non calendaires comme `q80j`, `M5` ou `noRef_...` restent ignorés pour le calcul automatique.


In [12]:
def parse_list_cell(x):
    """Transforme une cellule contenant une liste en vraie liste Python."""
    if pd.isna(x):
        return []
    if isinstance(x, list):
        return [str(v) for v in x]

    s = str(x).strip()
    if s in ["", "_", "[]"]:
        return []

    try:
        value = ast.literal_eval(s)
        if isinstance(value, list):
            return [str(v) for v in value]
        return [str(value)]
    except Exception:
        return [s]


def parse_time_code(code, texte=""):
    """Lit un COREF_name temporel, même incomplet."""
    parts = re.split(r"[-_]", str(code).strip())

    out = {
        "date_jour": np.nan,
        "jour_romanesque": np.nan,
        "time_precision": 0,
    }

    if len(parts) == 0 or not parts[0].isdigit():
        return pd.Series(out)

    annee = int(parts[0])
    if annee == 0:
        return pd.Series(out)

    # Année seule : exemple 1872-00-00-00.
    out["time_precision"] = 1

    if len(parts) < 2 or not parts[1].isdigit():
        return pd.Series(out)

    mois = int(parts[1])
    if mois == 0:
        return pd.Series(out)

    # Année + mois, mais pas forcément le jour.
    out["time_precision"] = 2

    if len(parts) < 3 or not parts[2].isdigit():
        return pd.Series(out)

    jour = int(parts[2])
    if jour == 0:
        return pd.Series(out)

    try:
        date = pd.Timestamp(year=annee, month=mois, day=jour)
    except Exception:
        return pd.Series(out)

    # Date complète : on peut calculer le jour romanesque.
    out["time_precision"] = 3

    if len(parts) >= 4 and parts[3].isdigit():
        heure = int(parts[3])
        if heure != 0 or "minuit" in str(texte).lower():
            out["time_precision"] = 4

    if len(parts) >= 5 and parts[4].isdigit():
        out["time_precision"] = max(out["time_precision"], 5)

    out["date_jour"] = date.strftime("%Y-%m-%d")
    out["jour_romanesque"] = (date - pd.Timestamp(DATE_DEBUT_CALENDRIER)).days + 1

    return pd.Series(out)

## 3) Préparer les temps

On garde les mentions `TIME` qui contiennent au moins une année exploitable. Une date complète permet de calculer le jour romanesque ; une année seule sert seulement de repère temporel.


In [13]:
# Sélection des entités temporelles.
time_mentions = ent_manuel[ent_manuel["cat"] == "TIME"].copy()
time_mentions = time_mentions.reset_index().rename(columns={"index": "entity_id"})

# Lecture des codes temporels contenus dans COREF_name.
parsed_time = time_mentions.apply(
    lambda row: parse_time_code(row["COREF_name"], row["text"]),
    axis=1,
)

time_mentions = pd.concat([time_mentions, parsed_time], axis=1)

# On garde tous les temps reconnus, même incomplets.
# Exemple : 1872-00-00-00 est gardé comme repère annuel,
# mais il n'a pas de date_jour ni de jour_romanesque.
time_abs = time_mentions[time_mentions["time_precision"] > 0].copy()
time_abs = time_abs.sort_values("start_token")

# Table de correspondance pour récupérer ensuite le texte et le code temporel.
time_lookup = time_mentions.set_index("entity_id")

time_abs[["text", "COREF_name", "date_jour", "jour_romanesque"]].head(10)

,text,COREF_name,date_jour,jour_romanesque
0,l' année 1872,1872-00-00-00,NaN,NaN
1,en 1814,1814-00-00-00,NaN,NaN
2,depuis de longues années,1862-00-00-00,NaN,NaN
3,chaque jour,1872-00-00-10,NaN,NaN
4,à des heures chronométriquement déterminées,1872-00-00-12-17,NaN,NaN
5,à minuit précis,1872-00-00-12,NaN,NaN
11,ce jour -là même,1872_10_02_00,1872-10-02,1.0
12,2 octobre,1872_10_02_00,1872-10-02,1.0
13,entre onze heures et onze heures et demie,1872_10_02_11,1872-10-02,1.0
14,à onze heures et demie sonnant,1872_10_02_11,1872-10-02,1.0


## 4) Rattacher un temps à chaque triptyque

Règle :

1. prendre un temps dans la même phrase si elle existe ;
2. sinon, prendre le dernier temps reconnu avant le verbe.

Amélioration par rapport à la version précédente : dans une même phrase, si plusieurs temps existent, on privilégie d’abord le temps directement présent dans le sujet ou l’objet du triptyque. Cela évite par exemple que `en 1814` contamine toute la première phrase.


In [14]:
# Dictionnaire : pour chaque phrase, liste des temps présents dans cette phrase.
time_by_sentence = {}

for key, sub in time_abs.groupby(["paragraph_ID", "sentence_ID"]):
    records = []
    for _, r in sub.sort_values("start_token").iterrows():
        start_sent = r.get("start_token_ID_within_sentence", np.nan)
        mention_len = r.get("mention_len", np.nan)

        if pd.isna(start_sent) or pd.isna(mention_len):
            span_ids = set()
        else:
            start_sent = int(start_sent)
            mention_len = int(mention_len)
            span_ids = set(range(start_sent, start_sent + mention_len))

        records.append({
            "entity_id": r["entity_id"],
            "start_token": r["start_token"],
            "end_token": r["end_token"],
            "start_sent": start_sent if not pd.isna(start_sent) else np.nan,
            "span_ids": span_ids,
            "date_jour": r["date_jour"],
            "time_precision": r["time_precision"],
        })
    time_by_sentence[key] = records


def token_ids_from_triplet(row):
    """Récupère les tokens du sujet, du verbe et de l'objet du triptyque."""
    ids = set()

    for col in ["IDs_sujet", "IDs_objet"]:
        for x in parse_list_cell(row.get(col, [])):
            if str(x).isdigit():
                ids.add(int(x))

    for col in ["ID_sujet", "ID_verbe", "ID_objet"]:
        x = row.get(col, np.nan)
        if pd.notna(x):
            ids.add(int(x))

    return ids


def time_in_same_sentence(row):
    """Renvoie le temps le plus pertinent dans la même phrase que le triptyque."""
    candidates = time_by_sentence.get((row["Num_paragr"], row["Num_phrase"]), [])
    if len(candidates) == 0:
        return np.nan

    # Priorité 1 : le temps est directement dans le sujet ou l'objet du triptyque.
    triplet_tokens = token_ids_from_triplet(row)
    direct = []
    for c in candidates:
        overlap = len(triplet_tokens & c["span_ids"])
        if overlap > 0:
            direct.append((overlap, c["time_precision"], c["start_token"], c))

    if len(direct) > 0:
        direct = sorted(direct, key=lambda x: (x[0], x[1], x[2]))
        return direct[-1][3]["entity_id"]

    # Priorité 2 : un temps placé au début de la phrase sert souvent de cadre général.
    scene_times = [c for c in candidates if pd.notna(c["start_sent"]) and c["start_sent"] <= 5]
    if len(scene_times) > 0:
        return scene_times[0]["entity_id"]

    # Priorité 3 : sinon, on prend le dernier temps situé avant le verbe.
    token_verbe = row["DocID_verbe"]
    before = [c for c in candidates if c["start_token"] <= token_verbe]

    if len(before) > 0:
        chosen = before[-1]

        # Cas fréquent : une heure précise est suivie par une date de rappel.
        # Exemple : "onze heures vingt-neuf du matin, ce mercredi 2 octobre 1872".
        if chosen["time_precision"] <= 3 and len(before) >= 2:
            better = [
                c for c in before[:-1]
                if c["date_jour"] == chosen["date_jour"]
                and c["time_precision"] > chosen["time_precision"]
                and chosen["start_token"] - c["start_token"] <= 15
            ]
            if len(better) > 0:
                chosen = better[-1]

        return chosen["entity_id"]

    # Si tous les temps sont après le verbe, on prend le premier temps de la phrase.
    return candidates[0]["entity_id"]


# Pour propager le temps d'une phrase à l'autre, on évite les dates parenthétiques.
# Si une phrase commence par un temps, ce temps est gardé comme cadre de la phrase.
context_records = []
for key, records in time_by_sentence.items():
    scene_times = [c for c in records if pd.notna(c["start_sent"]) and c["start_sent"] <= 5]
    if len(scene_times) > 0:
        context_records.append(scene_times[0])
    else:
        context_records.extend(records)

context_records = sorted(context_records, key=lambda x: x["start_token"])
starts = [c["start_token"] for c in context_records]
time_ids = [c["entity_id"] for c in context_records]


def previous_time_id(token_verbe):
    """Renvoie le dernier temps de contexte rencontré avant le verbe."""
    pos = bisect.bisect_right(starts, token_verbe) - 1
    if pos < 0:
        return np.nan
    return time_ids[pos]


def pick_time(entity_id, col):
    """Récupère une information temporelle à partir de l'identifiant d'entité."""
    if pd.isna(entity_id):
        return np.nan
    entity_id = int(entity_id)
    if entity_id not in time_lookup.index:
        return np.nan
    return time_lookup.loc[entity_id, col]


# Temps trouvé dans la phrase du triptyque.
trip["time_entity_id_phrase"] = trip.apply(time_in_same_sentence, axis=1)

# Dernier temps connu avant le verbe du triptyque.
trip["time_entity_id_precedent"] = trip["DocID_verbe"].apply(previous_time_id)

# Temps retenu automatiquement : phrase d'abord, sinon temps précédent.
trip["time_entity_id_auto"] = trip["time_entity_id_phrase"].combine_first(
    trip["time_entity_id_precedent"]
)

# Création des colonnes temporelles exportées.
for prefix in ["phrase", "precedent", "auto"]:
    id_col = f"time_entity_id_{prefix}"
    trip[f"time_texte_{prefix}"] = trip[id_col].apply(lambda x: pick_time(x, "text"))
    trip[f"time_code_{prefix}"] = trip[id_col].apply(lambda x: pick_time(x, "COREF_name"))
    trip[f"date_jour_{prefix}"] = trip[id_col].apply(lambda x: pick_time(x, "date_jour"))
    trip[f"jour_romanesque_{prefix}"] = trip[id_col].apply(lambda x: pick_time(x, "jour_romanesque"))

# Indique si le temps retenu vient de la phrase ou du contexte précédent.
trip["source_temps_auto"] = "precedent"
trip.loc[trip["time_entity_id_phrase"].notna(), "source_temps_auto"] = "phrase"
trip.loc[trip["time_entity_id_auto"].isna(), "source_temps_auto"] = "aucun"

## 5) Ajouter les lieux et le déplacement

On ne fait pas d’analyse du verbe ici.

```text
déplacement = au moins un personnage + au moins un lieu dans le triptyque
```


In [15]:
# Catégories d'entités utilisées comme lieux
CATEGORIES_ESPACE = ["FAC", "LOC", "GPE", "VEH"]

# Référentiel minimal : COREF de lieu -> nom lisible du lieu
lieu_mentions = ent_manuel[ent_manuel["cat"].isin(CATEGORIES_ESPACE)].copy()
lieu_mentions["COREF"] = lieu_mentions["COREF"].astype(str)

lieu_ref = lieu_mentions.sort_values("start_token").drop_duplicates("COREF")

lieux_lookup = {}
for _, row in lieu_ref.iterrows():
    label = row["COREF_name"]
    if pd.isna(label) or str(label).strip() == "":
        label = row["text"]
    lieux_lookup[str(row["COREF"])] = str(label)


def lieux_labels(ids):
    """Transforme les identifiants de lieux du triptyque en noms lisibles."""
    out = []
    for x in ids:
        label = lieux_lookup.get(str(x))
        if label is not None and label not in out:
            out.append(label)
    return out


# Listes de personnages et de lieux déjà repérés dans les triptyques
personnages_list = trip["personnages_triptyque"].apply(parse_list_cell)
lieux_list = trip["lieux_triptyque"].apply(parse_list_cell)

# Lieux lisibles + définition simple du déplacement
trip["lieux_labels"] = lieux_list.apply(lieux_labels)
trip["a_personnage"] = personnages_list.apply(lambda x: len(x) > 0)
trip["a_lieu"] = lieux_list.apply(lambda x: len(x) > 0)
trip["deplacement"] = trip["a_personnage"] & trip["a_lieu"]


## 6) Exporter `triptyques_temps_lieux.csv`


In [16]:
# Colonnes conservées dans le CSV final
colonnes_export = [
    "triplet_id", "chapter", "Num_paragr", "Num_phrase", "Sentence_index",
    "Phrase", "Sujet", "Verbe", "Objet", "Lemme_verbe",
    "personnages_triptyque", "lieux_triptyque", "types_lieux_triptyque",
    "time_texte_phrase", "time_code_phrase", "date_jour_phrase", "jour_romanesque_phrase",
    "time_texte_precedent", "time_code_precedent", "date_jour_precedent", "jour_romanesque_precedent",
    "time_texte_auto", "time_code_auto", "date_jour_auto", "jour_romanesque_auto", "source_temps_auto",
    "lieux_labels", "a_personnage", "a_lieu", "deplacement",
]

# Export unique du notebook
trip_export = trip[colonnes_export].copy()
trip_export.to_csv(OUT_CSV, index=False, encoding="utf-8")

print("Fichier écrit :", OUT_CSV)
print("Triptyques :", len(trip_export))
print("Triptyques avec temps :", trip_export["time_code_auto"].notna().sum())
print("Déplacements :", trip_export["deplacement"].sum())


Fichier écrit : ../results/csv_triptyques/triptyques_temps_lieux.csv
Triptyques : 1255
Triptyques avec temps : 1255
Déplacements : 61


In [18]:
# Aperçu du résultat final
trip_export[[
    "Sujet", "Verbe", "Objet",
    "time_code_auto", "lieux_labels", "deplacement",
]].head(20)


,Sujet,Verbe,Objet,time_code_auto,lieux_labels,deplacement
0,NaN,portant,le numéro 7 de Saville-row Burlington Gardens-...,1872-00-00-00,"[maison_fogg, rue_fogg]",False
1,Sheridan,mourut,en 1814,1814-00-00-00,[],False
2,la maison,habitée,En l'année 1872,1872-00-00-00,[maison_fogg],False
3,la maison,habitée,par Phileas Fogg esq.,1872-00-00-00,[maison_fogg],True
4,la maison,habitée,l'un des membres les plus singuliers et les pl...,1872-00-00-00,"[maison_fogg, reform_club]",True
5,la maison,semblât,NaN,1872-00-00-00,[maison_fogg],False
6,NaN,prendre,à tâche,1872-00-00-00,[],False
7,NaN,faire,NaN,1872-00-00-00,[],False
8,qui,pût,NaN,1872-00-00-00,[],False
9,qui,attirer,l'attention,1872-00-00-00,[],False
